# Multi-class Classification Support

[![Slides](https://img.shields.io/badge/🦌-ReHLine-blueviolet)](https://rehline-python.readthedocs.io/en/latest/)


`PLQ_Ridge_Classifier` extends binary PLQ-ERM to multi-class problems via the `multi_class`
parameter, supporting two standard decomposition strategies.

**One-vs-Rest (OvR)** fits $K$ binary classifiers, one per class, each trained on the
full dataset with relabelled targets ($+1$ for the class, $-1$ for all others).
Prediction selects the class with the highest decision score:

$$\widehat{y} = \arg\max_{k \in \{1, \dots, K\}} f_k(\mathbf{x})$$

**One-vs-One (OvO)** fits $\binom{K}{2}$ binary classifiers, one per class pair $(i, j)$,
each trained only on samples belonging to those two classes.
Prediction uses majority voting across all pairwise classifiers:

$$\widehat{y} = \arg\max_{k \in \{1, \dots, K\}} \sum_{j \neq k} \mathbf{1}\bigl[f_{kj}(\mathbf{x}) > 0\bigr]$$

In both cases, each binary sub-problem is solved by a standard `PLQ_Ridge_Classifier`
with the same loss, regularizer, and solver settings passed to the parent estimator.

In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from rehline import plq_Ridge_Classifier


# generate data
X_mc, y_mc = make_classification(
    n_samples=10000, n_features=20, n_informative=10,
    n_classes=4, n_clusters_per_class=1, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X_mc, y_mc, test_size=0.2, random_state=42)

### One-vs-Rest (OvR)


In OvR, we train $K$ binary classifiers, one per class. Classifier $k$ learns to distinguish
class $k$ from all other classes. The final prediction selects the class whose classifier
reports the highest decision score:

$$\widehat{y} \;=\; \arg\max_{k \in \{1, \dots, K\}} f_k(\mathbf{x})$$

where $f_k(\mathbf{x})$ is the signed distance from $\mathbf{x}$ to the decision boundary of classifier $k$.

In [3]:
# predict using ovr method
plq_ovr = plq_Ridge_Classifier(
    loss={'name': 'svm'}, C=1.0,
    fit_intercept=True, max_iter=50000, multi_class='ovr',
)
plq_ovr.fit(X_train, y_train)

y_pred = plq_ovr.predict(X_test)
print(f"plq OvR accuracy: {accuracy_score(y_test, y_pred):.4f}")

plq OvR accuracy: 0.7770


### One-vs-One (OvO)

In OvO, we train $\binom{K}{2}$ binary classifiers, one for each pair of classes $(i, j)$.
Each classifier $f_{ij}$ votes for either class $i$ or class $j$. The final prediction
is the class that receives the most votes:

$$\widehat{y} = \arg\max_{k \in \{1, \dots, K\}} \sum_{j \neq k} \mathbf{1}\bigl[f_{kj}(\mathbf{x}) > 0\bigr]$$

where $\mathbf{1}[\cdot]$ is the indicator function, and $f_{kj}(\mathbf{x}) > 0$ means
classifier $(k, j)$ votes for class $k$.

In [4]:
# predict using ovo method
plq_ovo = plq_Ridge_Classifier(
    loss={'name': 'svm'}, C=1.0,
    fit_intercept=True, max_iter=50000, multi_class='ovo',
)
plq_ovo.fit(X_train, y_train)
y_pred = plq_ovo.predict(X_test)

print(f"plq OvO accuracy: {accuracy_score(y_test, y_pred):.4f}")

plq OvO accuracy: 0.8035
